# Movement Model
The movement model simulates potential travel corridors under varying topographic conditions using a Monte Carlo framework and a two-scale cost-distance approach.
## Input
### Terrain and Cost Data
- High-resolution DEM (25 m)
- Coarse-resolution DEM (200 m)
- Cost raster (25 m and 200 m resolution)
- Barrier raster (25 m and 200 m resolution)
- Aspect raster (derived per iteration)
- Vertical factor table (slope-dependent movement cost)
### Access Points
- Fixed entrance points (entrancepoints)
- Randomly generated access points within a buffer around Lake Geneva (Buffer_1000m_LakeGeneva)
## Method
The model combines stochastic terrain perturbation with anisotropic cost-distance modelling in two spatial resolutions.
### 1. Monte Carlo Perturbation
For each of 100 iterations:
- Five random access points are generated within the Lake Geneva buffer (1000 m from the shore).
- A normally distributed elevation noise (σ = 1.5 m) is added to the 25 m DEM.
- The perturbed DEM is resampled to 200 m resolution.
- Aspect rasters are derived at both 25 m and 200 m resolution.
This introduces variability in terrain representation and resulting movement pathways.
### 2. Two-Scale Cost-Distance Modelling
For each access point within an iteration:
#### Coarse Scale (200 m)
- An anisotropic cost-distance accumulation is calculated using:
  - Slope-dependent vertical factor
  - Aspect-dependent horizontal factor
  - Cost raster
  - Barrier raster
- Optimal paths are calculated from all other points back to the source.
- The resulting paths are buffered (750 m) to define a movement corridor.
This step limits fine-scale computation to a plausible movement region.
#### Fine Scale (25 m)
Within the buffered corridor mask:
- A second anisotropic cost-distance accumulation is computed.
- Final optimal paths are derived at 25 m resolution.
The resulting polylines represent simulated movement routes between access points.
### 3. Aggregation
After all iterations:
- Each simulated path is buffered (50 m).
- Buffered paths are converted to binary rasters (1 = path presence).
- All rasters are summed.
- The summed raster is divided by the total number of simulated paths.
The final output raster therefore represents, for each cell, the relative frequency of simulated movement paths passing through that location.

## Implementation instructions
- Add all data under *data/01_Covariate/01.3_Movement/Movement.gdb* to the current map
- Indicate the link to the vertical factor (*cart.csv*) in the script
- Indicate the link to the *fact_cache folder*
- Adjust input manually (number of iterations, Distance Accumulation)
- Execute the cell

In [1]:
# Movement Model
import arcpy, os
from arcpy.sa import *

# ----------------------- CONFIGURATION -----------------------
# Define inputs
DEM25 = r"DEM"
COST25 = r"link_to_fast_cache\fast_cache\cost_25m.tif"
BARR25 = r"link_to_fast_cache\fast_cache\barrier_25m.tif"
DEM200 = r"DEM_200m"
COST200 = r"link_to_fast_cache\fast_cache\cost_200m.tif"
BARR200 = r"link_to_fast_cache\fast_cache\barrier_200m.tif"

# set parallel cores
arcpy.env.parallelProcessingFactor = "100%"

# define digital elevation model
in_raster = DEM25

# create describe object
desc = arcpy.Describe(arcpy.Describe(in_raster).path)
print(desc.path)

#configure environment settings
arcpy.env.overwriteOutput = True
arcpy.env.pyramid = "NONE"
arcpy.env.compression = "NONE"
arcpy.env.addOutputsToMap = False
arcpy.env.snapRaster = DEM200
arcpy.env.extent = DEM200
arcpy.env.cellSize = DEM200

# create new file geodatabase and define it as workspace
arcpy.management.CreateFileGDB(desc.path,"movement_rasters.gdb")
GDB = os.path.join(desc.path,"movement_rasters.gdb")
arcpy.management.CreateFolder(desc.path, "tmp")
TMP = os.path.join(desc.path,"tmp")
arcpy.env.workspace = GDB
OUT_GDB = GDB

# ----------------------- MODELLING -----------------------
# Monte Carlo specifics
N_RANDOM = 5
MC_SIGMA_M = 1.5                

# set vertical factor 
VF_TABLE = r"C:/Users/ufg/Documents/ArcGIS/Projects/Hilfsprojekt/Cart.txt"
vfact = VfTable(VF_TABLE)
HF_STRING = "LINEAR 0.5 181 1.11111111111111E-02"

# set stable parameters
Fixed_PTS = r"entrancepoints"
Genfersee = r"Buffer_1000m_LakeGeneva"

#loop
i = 100 #<-- number of iterations
for x in range (i):
    #create the 5 random points
    rnd_fc = "in_memory/random_pts"
    if arcpy.Exists("in_memory"): arcpy.management.Delete("in_memory")
    arcpy.management.CreateRandomPoints(out_path = os.path.dirname(rnd_fc),
                                        out_name = os.path.basename(rnd_fc),
                                        constraining_feature_class = Genfersee,
                                        number_of_points_or_field = N_RANDOM,
                                        minimum_allowed_distance = "1000 Meters")
    
    pts_iter = os.path.join(GDB,f"access_points_iter_"+str(x))
    if arcpy.Exists(pts_iter): arcpy.management.Delete(pts_iter)
    if arcpy.Exists(Fixed_PTS) and int(arcpy.management.GetCount(Fixed_PTS)[0])>0:
        arcpy.management.Merge([Fixed_PTS,rnd_fc],pts_iter)
    else:
        arcpy.management.CopyFeatures(rnd_fc,pts_iter)
    arcpy.management.Delete("in_memory")
    
    #create random noise and the DEM and aspect rasters
    rnd = arcpy.management.CreateRandomRaster(out_path = TMP,
                                            out_name = f"rnd_dem_"+str(x)+".tif",
                                            distribution = f"NORMAL 0.0 {MC_SIGMA_M}",
                                            raster_extent = DEM25,
                                            cellsize = 25,
                                            build_rat = "DO_NOT_BUILD")
    
    help_raster = (Raster(DEM25) + Raster(rnd))
    dem25_iter = os.path.join(TMP,f"dem25_mc_"+str(x)+".tif")
    help_raster.save(dem25_iter)
    dem_mc = os.path.join(TMP, f"dem200_mc_"+str(x))
    arcpy.management.Resample(help_raster, dem_mc, "200", "BILINEAR")
    arcpy.management.Delete(rnd)
    aspect_mc = os.path.join(TMP,f"aspect200_mc_"+str(x)+".tif")
    aspect_fine = os.path.join(TMP,f"aspect25_mc_"+str(x)+".tif")
    
    Aspect(Raster(dem_mc)).save(aspect_mc)
    Aspect(help_raster).save(aspect_fine)
    
    # loop through the entrance points and create a coarse everywhere-to-everywhere model
    Cs = os.path.join(GDB, f"C_{x}")
    Bs = os.path.join(GDB, f"B_{x}")
    helper_points = os.path.join(TMP,f"helper_point_"+str(x))
    arcpy.management.CopyFeatures(pts_iter, helper_points)
    n = int(arcpy.management.GetCount(pts_iter).getOutput(0))
    fields = ['OID@']
    oid_name = arcpy.Describe(pts_iter).OIDFieldName
    
    with arcpy.da.SearchCursor(pts_iter, fields) as cursor:
        for (oid_i,) in cursor:
            srcL = os.path.join(TMP, "_src_i")
            if arcpy.Exists(srcL): arcpy.management.Delete(srcL)
            arcpy.management.MakeFeatureLayer(pts_iter, srcL, f"{oid_name} = {oid_i}")
            da_coarse = DistanceAccumulation(in_source_data = srcL,
                                        in_barrier_data = BARR200,
                                        in_surface_raster = dem_mc,
                                        in_cost_raster = COST200,
                                        in_vertical_raster = dem_mc,
                                        vertical_factor = vfact,
                                        in_horizontal_raster = aspect_mc,
                                        horizontal_factor = HF_STRING,
                                        out_back_direction_raster = Bs,
                                        distance_method = "PLANAR")
            da_coarse.save(Cs)
            dstL = os.path.join(GDB,"_dst_all")
            if arcpy.Exists(dstL): arcpy.management.Delete(dstL)
            arcpy.management.MakeFeatureLayer(pts_iter, dstL)
            out_fc = os.path.join(TMP,"OPAL_coarse")
            out_buffer = os.path.join(TMP,"buffer_coarse.shp")  
            arcpy.sa.OptimalPathAsLine(in_destination_data = dstL,
                                       in_distance_accumulation_raster = Cs,
                                       in_back_direction_raster = Bs,
                                       out_polyline_features = out_fc,
                                       destination_field = "OBJECTID",
                                       path_type = "EACH_ZONE",
                                       create_network_paths = "DESTINATIONS_TO_SOURCES")
            
            # buffer the optimal paths to get corridors
            arcpy.analysis.PairwiseBuffer(out_fc, out_buffer, "750 Meters",dissolve_option = "ALL")
            mask25 = os.path.join(TMP,"mask.tif")
            tmp = os.path.join(TMP,"tmp.tif")
            field_buf = arcpy.Describe(out_buffer).OIDFieldName
            with arcpy.EnvManager(snapRaster = DEM25, cellSize = DEM25):
                arcpy.conversion.PolygonToRaster(out_buffer,field_buf, tmp, cellsize = DEM25)
            arcpy.sa.SetNull(IsNull(Raster(tmp)), 1).save(mask25)
            arcpy.management.Delete(tmp)
            arcpy.management.Delete(Bs)
            arcpy.management.Delete(Cs)

            #loop through the entrance points to create a fine model
            with arcpy.EnvManager(snapRaster = DEM25, cellSize = DEM25, mask = mask25, extent = mask25):
                da = DistanceAccumulation(in_source_data = srcL,
                                          in_barrier_data = BARR25,
                                          in_surface_raster = dem25_iter,
                                          in_cost_raster = COST25,
                                          in_vertical_raster = dem25_iter,
                                          vertical_factor = vfact,
                                          in_horizontal_raster = aspect_fine,
                                          horizontal_factor = HF_STRING,
                                          out_back_direction_raster = Bs,
                                          distance_method = "PLANAR")
            da.save(Cs)
            width = len(str(n))
            name = f"Iter{x}_OPAL{oid_i:0{width}d}"
            out_fc_def = os.path.join(GDB, name)
            arcpy.sa.OptimalPathAsLine(in_destination_data = dstL,
                                       in_distance_accumulation_raster = Cs,
                                       in_back_direction_raster = Bs,
                                       out_polyline_features = out_fc_def,
                                       destination_field = "OBJECTID",
                                       path_type = "EACH_ZONE",
                                       create_network_paths = "DESTINATIONS_TO_SOURCES")
            arcpy.management.Delete(Bs)
            arcpy.management.Delete(Cs)
            arcpy.management.Delete(out_fc)
            arcpy.management.Delete(dstL)
            arcpy.management.Delete(srcL)
            print(name)

# ----------------------- AGGREGATION -----------------------
print('')
print("Agregating results...")

# create a list of all features
in_paths=arcpy.ListFeatureClasses("Iter*","All")
print("list created")

# buffer features and transfrom into raster
buffered_list = []
buffer_distance = "50 Meters"

for fc in in_paths:
    buffer_output = os.path.join(GDB, f"buf_{fc}")
    arcpy.Buffer_analysis(fc, buffer_output, buffer_distance)
    field_names = [f.name for f in arcpy.ListFields(buffer_output)]
    if "ONE" not in field_names:
        arcpy.management.AddField(buffer_output, "ONE", "SHORT")
        arcpy.management.CalculateField(buffer_output, "ONE", 1)
    
    buffered_list.append(buffer_output)

raster_list = []
cell_size = 25

for buffered_fc in buffered_list:
    raster_output = os.path.join(GDB, f"ras_{os.path.basename(buffered_fc)}")
    arcpy.PolygonToRaster_conversion(buffered_fc, "ONE", raster_output, cell_assignment = "CELL_CENTER", priority_field="ORIG_FID", cellsize=cell_size)
    raster_list.append(raster_output)

sum_raster = os.path.join(GDB, "sum_occurrence")
out_sum = CellStatistics(raster_list, statistics_type = "SUM", ignore_nodata = "DATA")
out_sum.save(sum_raster)

# aggregate all rasters and divide them by the number of runs
total_rasters = len(raster_list)
relative_raster = os.path.join(GDB, "Movement") 
out_rel = Raster(sum_raster) / total_rasters
out_rel.save(relative_raster)
print("completed")

C:\Users\ufg\Documents\ArcGIS\Projects\Hilfsprojekt
Iter0_OPAL1
Iter0_OPAL2
Iter0_OPAL3
Iter0_OPAL4
Iter0_OPAL5
Iter0_OPAL6
Iter1_OPAL1
Iter1_OPAL2
Iter1_OPAL3
Iter1_OPAL4
Iter1_OPAL5
Iter1_OPAL6
﻿
Agregating results...
list created
completed
